# FD004 Paper 2, 60-cycle context: standard (IID) evaluation, five seeds, converged training

Paper 2 main experiment. Identical to the Paper 1 rerun notebooks (06–08) except the input window: **60 cycles instead of 30**. The DS-A rerun (`06_FD004_DS_A_Rerun_Converged_FiveSeed.ipynb`) trains under a pre-specified,
training-only convergence rule. This notebook applies **the same rule** to the IID protocol so that IID, DS-A and DS-B are
reported on one training protocol, five seeds each (42, 123, 456, 789, 1000), seed 42 as the single-run reference.

Unchanged from the frozen IID pipeline: data split, preprocessing, windows, architecture (20→18 projection, BiLSTM(64),
temporal attention, q10/q50/q90 pinball loss), Adam 1e-3, batch 256, and every calibration rule.

Changed (Part A, written before any training): 20% of training engines (split seed 42) become internal validation; up to
150 epochs; early stopping on internal-validation pinball loss, patience 15, not before epoch 30, best weights restored;
converged if best validation loss ≤ 0.60 × constant-quantile loss. Every seed is reported; none is replaced.

Part B trains and saves all models without touching evaluation windows; Part C evaluates afterwards.
Set `FD004_SMOKE=1` for a 1-seed, 3-epoch pipeline check.

In [1]:
import os, sys, json, time, random, hashlib, platform
from pathlib import Path

SMOKE_TEST = os.environ.get("FD004_SMOKE") == "1"

def _find_root():
    for c in [os.environ.get("FD004_ROOT", ""), "/content/drive/MyDrive/FD004_Research", str(Path.home() / "Downloads/FD004_Research")]:
        if c and Path(c, "CMAPSSData/train_FD004.txt").exists():
            return Path(c)
    raise FileNotFoundError("Set FD004_ROOT to the FD004_Research folder.")

if "google.colab" in sys.modules and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = _find_root()
SLUG = "IID_W60"
CONDITION = "IID"
W = 60

OUT = Path(os.environ.get("FD004_OUT") or ROOT / (f"FD004_{SLUG}_Results" + ("_SMOKE" if SMOKE_TEST else "")))
(OUT / "models").mkdir(parents=True, exist_ok=True)
(OUT / "histories").mkdir(exist_ok=True)
print("Root:", ROOT, "| Output:", OUT)

Root: <FD004_ROOT> | Output: <FD004_ROOT>/FD004_IID_W60_Results


## Part A: protocol (written before any training or evaluation)

In [2]:
PROTOCOL = {
    "experiment": f"FD004 Paper 2 {CONDITION}, {W}-cycle context, converged five-seed run (Paper 1 rule)",
    "window_size": W,
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "data_seed": 42,
    "model_seeds": [42] if SMOKE_TEST else [42, 123, 456, 789, 1000],
    "internal_validation": {"fraction_of_train_engines": 0.20, "split_seed": 42},
    "model": {"projection": 18, "bilstm_units": 64, "quantiles": [0.10, 0.50, 0.90], "learning_rate": 1e-3, "batch_size": 256},
    "training_rule": {"max_epochs": 3 if SMOKE_TEST else 150, "monitor": "val_loss", "patience": 15,
                      "start_from_epoch": 0 if SMOKE_TEST else 30, "restore_best_weights": True},
    "convergence_check": {"ratio": 0.60, "reference": "constant training quantiles evaluated on internal validation",
                          "failed_seeds": "reported and flagged; never replaced or dropped"},
    "calibration": {"alpha": 0.20, "methods": ["Raw Quantiles", "Window CQR", "Asymmetric CQR",
                                               "Regime-conditional Asymmetric CQR", "Balanced Engine CQR"],
                    "window": "finite-sample higher quantile at ceil((n+1)(1-alpha))/n",
                    "asymmetric": "per tail at ceil((n+1)(1-alpha/2))/n, unclipped scores",
                    "regime_conditional": "asymmetric rule within each endpoint regime",
                    "balanced": "within-engine np.quantile(.,0.80), then ceil((M+1)(1-alpha))/M higher",
                    "interval_clip_at_zero": True},
    "life_stage": {"Early": "RUL > 80", "Mid": "40 < RUL <= 80", "Late": "RUL <= 40"},
    "bootstrap": {"replicates": 10000, "unit": "evaluation engine", "rng_seed": 42},
}
protocol_path = OUT / "protocol.json"
if protocol_path.exists() and not SMOKE_TEST:
    old = json.loads(protocol_path.read_text())
    changed = {k for k in PROTOCOL if k != "created_utc" and old.get(k) != PROTOCOL[k]}
    assert not changed, f"Protocol already registered and differs in {changed}."
    PROTOCOL["created_utc"] = old["created_utc"]
else:
    protocol_path.write_text(json.dumps(PROTOCOL, indent=2))
print("Protocol sha256:", hashlib.sha256(protocol_path.read_bytes()).hexdigest()[:16])

Protocol sha256: 8e590ca174b3d9e3


## Data and windows (identical to the frozen pipeline)

In [3]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
P = PROTOCOL
b = np.load(ROOT / "artifacts/FD004_baseline_tensors.npz")
t = np.load(ROOT / "artifacts/P2_E8_60cycle_tensors.npz")
assert W == 60

def engine_ids(y):
    ids = np.ones(len(y), int); ids[1:] = 1 + np.cumsum(np.diff(y) > 0); return ids

# Paper 1 IID engine indices (order of appearance in the 30-cycle tensors) mapped to C-MAPSS units of the 60-cycle tensors.
id_tr = 1 + pd.Index(pd.unique(t["train_endpoint_unit"])).get_indexer(t["train_endpoint_unit"])
id_v = 1 + pd.Index(pd.unique(t["val_endpoint_unit"])).get_indexer(t["val_endpoint_unit"])
n30 = np.bincount(engine_ids(b["y_val"]))[1:]; n60 = np.bincount(id_v)[1:]
assert id_tr.max() == 200 and id_v.max() == 49 and np.all(n30 - n60 == 30), "60-cycle tensors do not align with the Paper 1 IID split"
Xtr_raw, ytr, Xv_raw, yv = t["X_train"], t["y_train"].astype(np.float32), t["X_val"], t["y_val"].astype(np.float32)

np.random.seed(42)
cal_engines = np.random.choice(np.unique(engine_ids(b["y_val"])), size=25, replace=False)
test_engines = np.setdiff1d(np.arange(1, 50), cal_engines)
s = np.load(ROOT / "models/b6_variants/b6_cal_eval_split_ids.npz")
assert set(s["cal_engine_ids"]) == set(cal_engines) and set(s["eval_engine_ids"]) == set(test_engines)

def onehot(X):
    return np.concatenate([X[:, :, :14], np.eye(6, dtype=np.float32)[X[:, :, 14].astype(int)]], axis=-1).astype(np.float32)

train_engines = np.unique(id_tr)
fit_engines, ival_engines = train_test_split(train_engines, test_size=P["internal_validation"]["fraction_of_train_engines"],
                                             random_state=P["internal_validation"]["split_seed"])
mf, mi = np.isin(id_tr, fit_engines), np.isin(id_tr, ival_engines)
mc, mt = np.isin(id_v, cal_engines), np.isin(id_v, test_engines)
X_fit, y_fit = onehot(Xtr_raw[mf]), ytr[mf]
X_iv, y_iv = onehot(Xtr_raw[mi]), ytr[mi]
X_cal, y_cal, id_cal, reg_cal = onehot(Xv_raw[mc]), yv[mc], id_v[mc], Xv_raw[mc][:, -1, 14].astype(int)
X_test, y_test, id_test, reg_test = onehot(Xv_raw[mt]), yv[mt], id_v[mt], Xv_raw[mt][:, -1, 14].astype(int)
cyc_test = t["val_endpoint_cycle"][mt]
assert len(y_cal) == 4909 - 25 * 30 and len(y_test) == 5423 - 24 * 30
print(f"Train engines {len(train_engines)} -> fit {len(fit_engines)} + internal validation {len(ival_engines)}")
print("fit", X_fit.shape, "| ival", X_iv.shape, "| cal", X_cal.shape, "| test", X_test.shape)
pd.concat([pd.DataFrame({"split": n, "engine_index": v}) for n, v in
           [("fit", fit_engines), ("internal_val", ival_engines), ("calibration", np.sort(cal_engines)), ("evaluation", test_engines)]]
          ).to_csv(OUT / "engine_split.csv", index=False)

Train engines 200 -> fit 160 + internal validation 40
fit (30183, 60, 20) | ival (7513, 60, 20) | cal (4159, 60, 20) | test (4703, 60, 20)


## Part B: train five seeds and check convergence (no evaluation data)

In [4]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LSTM, Bidirectional, Layer
from tensorflow.keras.models import Model
Q = P["model"]["quantiles"]

@tf.keras.utils.register_keras_serializable(package="fd004")
class FastTemporalAttention(Layer):
    def build(self, input_shape):
        d = input_shape[-1]
        self.W = self.add_weight(name="W", shape=(d, d), initializer="glorot_uniform")
        self.b = self.add_weight(name="b", shape=(d,), initializer="zeros")
        self.v = self.add_weight(name="v", shape=(d, 1), initializer="glorot_uniform")
        super().build(input_shape)
    def call(self, x):
        e = tf.nn.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        return tf.reduce_sum(x * tf.nn.softmax(tf.tensordot(e, self.v, axes=1), axis=1), axis=1)

@tf.keras.utils.register_keras_serializable(package="fd004")
def multi_quantile_pinball_loss(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, (-1, 1)), tf.float32)
    return tf.reduce_mean(tf.add_n([tf.maximum(q * (y_true - y_pred[:, i:i+1]), (q - 1.0) * (y_true - y_pred[:, i:i+1]))
                                    for i, q in enumerate(Q)]))

def build_model():
    inp = Input(shape=(W, 20), name="input_window")
    h = Dense(18, use_bias=False, name="projection")(inp)
    h = Bidirectional(LSTM(64, return_sequences=True), name="bilstm")(h)
    h = FastTemporalAttention(name="temporal_attention")(h)
    m = Model(inp, Dense(3, name="quantiles")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=multi_quantile_pinball_loss)
    return m

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s); os.environ["PYTHONHASHSEED"] = str(s)

const_iv = float(sum(np.mean(np.maximum(q * (y_iv - np.quantile(y_fit, q)), (q - 1) * (y_iv - np.quantile(y_fit, q)))) for q in Q))
print(f"Constant-quantile loss on internal validation: {const_iv:.3f}")
(OUT / "environment.json").write_text(json.dumps({"python": platform.python_version(), "platform": platform.platform(),
    "tensorflow": tf.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "gpu": [d.name for d in tf.config.list_physical_devices("GPU")]}, indent=2))

tr = P["training_rule"]; train_log = []
for seed in P["model_seeds"]:
    tf.keras.backend.clear_session(); set_seed(seed); model = build_model()
    stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=tr["patience"], start_from_epoch=tr["start_from_epoch"],
                                            restore_best_weights=True)
    t0 = time.time()
    hist = model.fit(X_fit, y_fit, validation_data=(X_iv, y_iv), epochs=tr["max_epochs"], batch_size=256, shuffle=True,
                     callbacks=[stop], verbose=2)
    h = pd.DataFrame(hist.history); h.insert(0, "epoch", np.arange(1, len(h) + 1))
    h.to_csv(OUT / "histories" / f"seed{seed}_history.csv", index=False)
    model.save(OUT / "models" / f"{SLUG}_rerun_seed{seed}.keras")
    best = float(h.val_loss.min())
    train_log.append({"seed": seed, "epochs_run": len(h), "best_epoch": int(h.loc[h.val_loss.idxmin(), "epoch"]),
                      "best_val_loss": best, "constant_val_loss": const_iv, "ratio": best / const_iv,
                      "converged": best <= P["convergence_check"]["ratio"] * const_iv, "minutes": (time.time() - t0) / 60})
    print(train_log[-1])
train_log = pd.DataFrame(train_log); train_log.to_csv(OUT / "convergence_check.csv", index=False)
print(train_log.round(3).to_string(index=False))

Constant-quantile loss on internal validation: 30.027


Epoch 1/150


118/118 - 62s - 524ms/step - loss: 106.3712 - val_loss: 91.7253


Epoch 2/150


118/118 - 48s - 407ms/step - loss: 83.5950 - val_loss: 75.4581


...

Epoch 49/150


118/118 - 17s - 142ms/step - loss: 5.9001 - val_loss: 14.0715


Epoch 50/150


118/118 - 20s - 168ms/step - loss: 5.9474 - val_loss: 14.7706


{'seed': 1000, 'epochs_run': 50, 'best_epoch': 27, 'best_val_loss': 11.648350715637207, 'constant_val_loss': 30.026634216308594, 'ratio': 0.38793394663297126, 'converged': True, 'minutes': 26.99560673236847}
 seed  epochs_run  best_epoch  best_val_loss  constant_val_loss  ratio  converged  minutes
   42          51          27         14.533             30.027  0.484       True   24.409
  123          46          28         18.184             30.027  0.606      False   24.467
  456          47          32         11.471             30.027  0.382       True   39.000
  789          46          26         10.498             30.027  0.350       True   34.495
 1000          50          27         11.648             30.027  0.388       True   26.996


## Part C: evaluation (run only after Part B is saved)

In [5]:
assert (OUT / "convergence_check.csv").exists()
from scipy.stats import pearsonr, spearmanr
A = P["calibration"]["alpha"]

def cq(scores, a):
    n = len(scores)
    return float(np.quantile(scores, min(1.0, np.ceil((n + 1) * (1 - a)) / n), method="higher"))

def stage_of(r):
    return np.where(r > 80, "Early", np.where(r > 40, "Mid", "Late"))

rows, glob_rows, eng_rows, stage_rows, corr_rows, corrections = [], [], [], [], [], []
for seed in P["model_seeds"]:
    model = tf.keras.models.load_model(OUT / "models" / f"{SLUG}_rerun_seed{seed}.keras")
    pc = model.predict(X_cal, batch_size=512, verbose=0); pt = model.predict(X_test, batch_size=512, verbose=0)
    lc, uc, lt, ut = pc[:, 0], pc[:, 2], pt[:, 0], pt[:, 2]
    qw = cq(np.maximum(lc - y_cal, y_cal - uc), A)
    ql, qu = cq(lc - y_cal, A / 2), cq(y_cal - uc, A / 2)
    err = np.maximum(lc - y_cal, y_cal - uc)
    eq = np.array([np.quantile(err[id_cal == e], 0.80) for e in np.unique(id_cal)]); M = len(eq)
    qb = float(np.quantile(eq, min(1.0, np.ceil((M + 1) * (1 - A)) / M), method="higher"))
    rl, ru = np.zeros_like(lt), np.zeros_like(ut)
    for r in (range(6) if CONDITION != "DS-A" else []):  # not applicable to DS-A: Regime 0 has no calibration windows
        mcal, mt = reg_cal == r, reg_test == r
        a1, a2 = cq(lc[mcal] - y_cal[mcal], A / 2), cq(y_cal[mcal] - uc[mcal], A / 2)
        rl[mt], ru[mt] = lt[mt] - a1, ut[mt] + a2
        corrections.append({"seed": seed, "method": "Regime-conditional", "regime": r, "q_lower": a1, "q_upper": a2})
    corrections += [{"seed": seed, "method": "Window", "q": qw}, {"seed": seed, "method": "Asymmetric", "q_lower": ql, "q_upper": qu},
                    {"seed": seed, "method": "Balanced", "q": qb}]
    methods = {"Raw Quantiles": (lt, ut), "Window CQR": (lt - qw, ut + qw), "Asymmetric CQR": (lt - ql, ut + qu),
               "Regime-conditional Asym. CQR": (rl, ru), "Balanced Engine CQR": (lt - qb, ut + qb)}
    if CONDITION == "DS-A":
        methods.pop("Regime-conditional Asym. CQR")
    df = pd.DataFrame({"seed": seed, "engine": id_test, "cycle": cyc_test, "endpoint_regime": reg_test, "true_RUL": y_test,
                       "q10": lt, "q50": pt[:, 1], "q90": ut, "stage": stage_of(y_test)})
    df["error_q50_minus_true"] = df.q50 - df.true_RUL
    for name, (lo, hi) in methods.items():
        lo, hi = np.maximum(0, lo), np.maximum(0, hi)
        k = name.split()[0].lower().replace("-", "_")
        df[f"{k}_lower"], df[f"{k}_upper"] = lo, hi
        cov = (y_test >= lo) & (y_test <= hi)
        wink = (hi - lo) + (2 / A) * (lo - y_test) * (y_test < lo) + (2 / A) * (y_test - hi) * (y_test > hi)
        ec = pd.Series(cov).groupby(id_test).mean() * 100
        glob_rows.append({"seed": seed, "method": name, "RMSE": float(np.sqrt(np.mean(df.error_q50_minus_true ** 2))),
                          "MAE": float(df.error_q50_minus_true.abs().mean()), "PICP": 100 * cov.mean(), "MPIW": float((hi - lo).mean()),
                          "Winkler": float(wink.mean()), "lower_violation": 100 * (y_test < lo).mean(), "upper_violation": 100 * (y_test > hi).mean(),
                          "eng_mean": ec.mean(), "eng_median": ec.median(), "eng_p10": np.percentile(ec, 10), "eng_p25": np.percentile(ec, 25),
                          "eng_worst": ec.min(), "below80": int((ec < 80).sum()), "below70": int((ec < 70).sum())})
        df[f"{k}_covered"] = cov
    for e, g in df.groupby("engine"):
        eng_rows.append({"seed": seed, "engine": e, "n": len(g), "RMSE": float(np.sqrt(np.mean(g.error_q50_minus_true ** 2))),
                         "MAE": float(g.error_q50_minus_true.abs().mean()), "bias": float(g.error_q50_minus_true.mean()),
                         "asym_cov": 100 * g.asymmetric_covered.mean(), "bal_cov": 100 * g.balanced_covered.mean(),
                         "bal_below_lower": 100 * (g.true_RUL < g.balanced_lower).mean(), "bal_above_upper": 100 * (g.true_RUL > g.balanced_upper).mean(),
                         "mid_bal_cov": 100 * g.balanced_covered[g.stage == "Mid"].mean() if (g.stage == "Mid").any() else np.nan,
                         "dominant_regime": int(g.endpoint_regime.mode().iloc[0]), "dominant_share": float((g.endpoint_regime == g.endpoint_regime.mode().iloc[0]).mean())})
    for s_, g in df.groupby("stage"):
        stage_rows.append({"seed": seed, "stage": s_, "N": len(g), "RMSE": float(np.sqrt(np.mean(g.error_q50_minus_true ** 2))),
                           "signed_bias": float(g.error_q50_minus_true.mean()), "asym_cov": 100 * g.asymmetric_covered.mean(),
                           "bal_cov": 100 * g.balanced_covered.mean()})
    pe = pd.DataFrame([r for r in eng_rows if r["seed"] == seed])
    corr_rows.append({"seed": seed, "pearson_r": pearsonr(pe.RMSE, pe.bal_cov)[0], "pearson_p": pearsonr(pe.RMSE, pe.bal_cov)[1],
                      "spearman_rho": spearmanr(pe.RMSE, pe.bal_cov)[0],
                      "problem_engines": " ".join(map(str, sorted(pe.engine[pe.bal_cov < 80])))})
    rows.append(df)

pred = pd.concat(rows, ignore_index=True); pred.to_csv(OUT / "test_predictions_rowlevel.csv", index=False)
glob = pd.DataFrame(glob_rows).merge(train_log[["seed", "converged"]], on="seed"); glob.to_csv(OUT / "global_by_seed.csv", index=False)
per_engine = pd.DataFrame(eng_rows); per_engine.to_csv(OUT / "per_engine_by_seed.csv", index=False)
stages = pd.DataFrame(stage_rows); stages.to_csv(OUT / "life_stage_by_seed.csv", index=False)
pd.DataFrame(corr_rows).to_csv(OUT / "rmse_coverage_correlation_by_seed.csv", index=False)
pd.DataFrame(corrections).to_csv(OUT / "corrections_by_seed.csv", index=False)
print(glob[glob.seed == P["model_seeds"][0]].round(2).to_string(index=False))

 seed                       method  RMSE   MAE  PICP  MPIW  Winkler  lower_violation  upper_violation  eng_mean  eng_median  eng_p10  eng_p25  eng_worst  below80  below70  converged
   42                Raw Quantiles 24.32 16.47 58.41 32.95    96.55            11.84            29.75     57.98       57.50    34.45    42.56      25.50       20       16       True
   42                   Window CQR 24.32 16.47 80.52 54.68    85.82             5.12            14.35     80.99       84.79    56.40    71.08      49.22        9        6       True
   42               Asymmetric CQR 24.32 16.47 81.86 56.20    83.02             7.08            11.06     81.40       87.04    57.81    74.32      47.66       10        6       True
   42 Regime-conditional Asym. CQR 24.32 16.47 81.86 56.42    83.47             7.00            11.14     81.41       86.94    58.82    72.97      46.88       10        6       True
   42          Balanced Engine CQR 24.32 16.47 90.81 79.51    91.51             1.93      

In [6]:
# Engine-cluster bootstrap and five-seed summary
boot = []
for seed in P["model_seeds"]:
    g = pred[pred.seed == seed]
    agg = g.groupby("engine").agg(n=("true_RUL", "size"), ca=("asymmetric_covered", "sum"), cb=("balanced_covered", "sum"))
    agg["wa"] = (g.asymmetric_upper - g.asymmetric_lower).groupby(g.engine).sum()
    agg["wb"] = (g.balanced_upper - g.balanced_lower).groupby(g.engine).sum()
    n = agg.n.to_numpy(); rng = np.random.default_rng(42); idx = rng.integers(0, len(n), size=(10000, len(n))); N = n[idx].sum(1)
    covA, covB = 100 * agg.ca.to_numpy() / n, 100 * agg.cb.to_numpy() / n
    S = {"asym_picp": (100 * agg.ca.sum() / n.sum(), 100 * agg.ca.to_numpy()[idx].sum(1) / N),
         "bal_picp": (100 * agg.cb.sum() / n.sum(), 100 * agg.cb.to_numpy()[idx].sum(1) / N),
         "asym_mpiw": (agg.wa.sum() / n.sum(), agg.wa.to_numpy()[idx].sum(1) / N),
         "bal_mpiw": (agg.wb.sum() / n.sum(), agg.wb.to_numpy()[idx].sum(1) / N),
         "asym_eng_mean": (covA.mean(), covA[idx].mean(1)), "bal_eng_mean": (covB.mean(), covB[idx].mean(1)),
         "asym_eng_p10": (np.percentile(covA, 10), np.percentile(covA[idx], 10, axis=1)),
         "bal_eng_p10": (np.percentile(covB, 10), np.percentile(covB[idx], 10, axis=1)),
         "asym_eng_worst": (covA.min(), covA[idx].min(1)), "bal_eng_worst": (covB.min(), covB[idx].min(1))}
    S["bal_minus_asym_picp"] = (S["bal_picp"][0] - S["asym_picp"][0], S["bal_picp"][1] - S["asym_picp"][1])
    for k, (pt_, v) in S.items():
        boot.append({"seed": seed, "stat": k, "point": pt_, "ci_low": np.percentile(v, 2.5), "ci_high": np.percentile(v, 97.5)})
pd.DataFrame(boot).to_csv(OUT / "bootstrap_by_seed.csv", index=False)

five = glob.groupby("method")[["PICP", "MPIW", "Winkler", "RMSE", "MAE", "below80", "eng_worst"]].agg(["mean", "std"])
five.columns = [f"{a}_{b}" for a, b in five.columns]; five.to_csv(OUT / "five_seed_summary.csv")
stages.groupby("stage")[["RMSE", "signed_bias", "bal_cov", "asym_cov"]].agg(["mean", "std"]).to_csv(OUT / "life_stage_five_seed_summary.csv")
print(five.round(2).to_string())
manifest = {str(p.relative_to(OUT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.rglob("*")) if p.is_file()}
(OUT / "manifest_sha256.json").write_text(json.dumps(manifest, indent=2)); print("Done:", len(manifest), "files")

                              PICP_mean  PICP_std  MPIW_mean  MPIW_std  Winkler_mean  Winkler_std  RMSE_mean  RMSE_std  MAE_mean  MAE_std  below80_mean  below80_std  eng_worst_mean  eng_worst_std
method                                                                                                                                                                                             
Asymmetric CQR                    82.74      1.38      50.04     10.43         74.70        10.44      21.21      3.66      13.9     2.84          10.0         2.12           45.66           7.95
Balanced Engine CQR               92.56      1.91      74.39     17.90         84.48        15.29      21.21      3.66      13.9     2.84           3.6         1.67           63.16           6.43
Raw Quantiles                     64.44      4.85      32.12     10.01         80.52        13.91      21.21      3.66      13.9     2.84          18.6         2.97           22.53           8.93
Regime-conditional A